In [1]:
import cvxpy as cp
import numpy as np

def QMC_SDP_solver(edges, weights, n_vertices):
    # Step 1: Declaring M as a variable
    M = cp.Variable((n_vertices, n_vertices), symmetric=True)

    # Step 2: Set up of the objective function
    objective = 0
    for (i, j), w in zip(edges, weights):
        objective += w * (1 - M[i, j]) / 2

    # Step 3: Defining constraints
    # 3.3 M PSD:
    constraints = [
        M >> 0,
    ]

    # 3.1 Diagonal entries normalised to 1/enforcing identity for products of equal pauli operators, i.e. p^+ p = I
    for i in range(n_vertices):
        constraints.append(M[i, i] == 1)

    # Can neglect second constraint (3.2 here) since for the max cut setting it is never the case that we have one pauli operator on the one qubit but a different one on the other; We always only ever apply Pauli-z in QMC

    # Step 4: Create and solve the problem
    problem = cp.Problem(cp.Maximize(objective), constraints)
    problem.solve()

    # Step 5: Return the optimal M found by the solver
    return M.value

if __name__ == '__main__':
    edges = [(0, 1), (0, 3), (2, 1), (2, 3), (0, 2)]  # A triangle graph
    weights = [1, 1, 1, 1, 1]
    n_vertices = 4

    M_optimal = QMC_SDP_solver(edges, weights, n_vertices)
    print("Optimal moment matrix:")
    print(M_optimal)

Optimal moment matrix:
[[ 0.99999999 -0.99982574  0.99930305 -0.99982574]
 [-0.99982574  0.99999999 -0.99982574  0.99999996]
 [ 0.99930305 -0.99982574  0.99999999 -0.99982574]
 [-0.99982574  0.99999996 -0.99982574  0.99999999]]


In [2]:
import random

In [3]:
def random_instance_generator(nodes: int, weights_static: bool):
    edges = []
    weights = []
    avg_n_edges = random.random() # random threshold for edge generation (directly correlating to the average number of edges generated among the existing vertices)
    for i in range(nodes):
        for j in range(nodes):
            if i != j and random.random() > avg_n_edges:
                edges.append((i, j))
                weights.append(random.uniform(1e-10, 1.0) if not weights_static else 1)

    return edges, weights, nodes


In [4]:
def QMC_rounding(M: cp.Variable):
    n_vertices = M.shape[0]

In [5]:
edges, weights, n_vertices = random_instance_generator(50, True)

print(f'number of vertices = {n_vertices}, number of edges = {len(edges)}')

M_optimal = QMC_SDP_solver(edges, weights, n_vertices)
print(f"Optimal moment matrix: with dimension: {M_optimal.shape}")
print(M_optimal)

number of vertices = 50, number of edges = 2206


/opt/homebrew/Caskroom/miniconda/base/envs/MasterThesis/lib/python3.11/site-packages/cvxpy/problems/problem.py:167: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn("Objective contains too many subexpressions. "


Optimal moment matrix: with dimension: (50, 50)
[[ 1.          0.23635009  0.94388896 ... -0.83925676 -0.93563972
  -0.2510707 ]
 [ 0.23635009  1.         -0.04629795 ... -0.63715994 -0.31501064
   0.80255859]
 [ 0.94388896 -0.04629795  1.         ... -0.61261311 -0.91728347
  -0.5524467 ]
 ...
 [-0.83925676 -0.63715994 -0.61261311 ...  1.          0.7260647
  -0.30720908]
 [-0.93563972 -0.31501064 -0.91728347 ...  0.7260647   1.
   0.28075684]
 [-0.2510707   0.80255859 -0.5524467  ... -0.30720908  0.28075684
   1.        ]]


In [6]:
def cholesky_decomposition(A):
    """
    Compute Cholesky decomposition (via Cholesky-Banachiewicz algorithm) of Hermitian PSD matrix A.
    Returns lower triangular matrix L such that A = L @ L.conj().T
    """
    A = (A + A.conj().T) / 2
    n = A.shape[0]
    L = np.zeros_like(A)

    for i in range(n):
        for j in range(i + 1):
            if i == j:
                # Diagonal element
                sum_val = np.sum(L[i, :j] * L[i, :j].conj())
                L[i, i] = np.sqrt(A[i, i] - sum_val)
            else:
                # Off-diagonal element
                sum_val = np.sum(L[i, :j] * L[j, :j].conj())
                L[i, j] = (A[i, j] - sum_val) / L[j, j]

    return L

In [7]:
def cholesky_psd(A, eps=1e-12):
    """
    Robust Cholesky for Hermitian PSD matrices that may have small
    negative eigenvalues from numerical noise.

    Steps:
      1) Symmetrize A
      2) Project to PSD by clipping eigenvalues
      3) Standard Cholesky
    """
    # 1. Force exact Hermitian
    A = (A + A.conj().T) / 2

    # 2. Eigenvalue repair
    w, V = np.linalg.eigh(A)
    w_clipped = np.maximum(w, eps)   # eliminate small negative pivots
    A_psd = V @ np.diag(w_clipped) @ V.conj().T

    # 3. Cholesky on repaired matrix
    return np.linalg.cholesky(A_psd)

In [8]:
print(cholesky_psd(M_optimal))

[[ 9.99999999e-01  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 2.36350086e-01  9.71667965e-01  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 9.43888961e-01 -2.77240987e-01  1.79474405e-01 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [-8.39256757e-01 -4.51596171e-01  3.02834877e-01 ...  1.75249252e-06
   0.00000000e+00  0.00000000e+00]
 [-9.35639723e-01 -9.66092454e-02 -3.39477400e-01 ... -2.65385958e-07
   1.33602058e-06  0.00000000e+00]
 [-2.51070701e-01  8.87030553e-01 -3.87479287e-01 ... -2.75667839e-07
   1.26547455e-08  1.90029143e-06]]


In [9]:
def round_sdp_with_cholesky(M, num_rounds=1):
    """
    Round an SDP solution using Cholesky decomposition.

    The key insight: if M = L L^T, then the columns of L
    are vectors v_i such that M[i,j] = v_i · v_j
    """
    n = M.shape[0]

    # Get Cholesky decomposition: M = L L^T
    L = cholesky_psd(M)

    # The vectors are the rows of L^T (or columns of L)
    # Each row i of L^T is the vector for vertex i
    V = L.T

    # Verify our decomposition (optional check)
    # print(f"Reconstruction error: {np.max(np.abs(V.T @ V - M)):.6f}")

    cuts = []
    for _ in range(num_rounds):
        # Random hyperplane rounding
        # Pick a random direction in the embedding space
        random_normal = np.random.randn(V.shape[1])
        random_normal /= np.linalg.norm(random_normal)

        # Project each vector onto this direction
        projections = V @ random_normal

        # Partition based on sign
        partition = (projections > 0).astype(int)
        cuts.append(partition)

    return cuts[0] if num_rounds == 1 else cuts

In [10]:
import os
os.environ['GRB_LICENSE_FILE'] = '/Users/julian/PycharmProjects/PythonProject/MasterThesis/gurobi.lic'
import gurobipy as gp
from gurobipy import Model, GRB, quicksum

def gurobi_maxcut(n, edges, weights, time_limit=None, mip_gap=None, verbose=True):
    """
    Solve Max-Cut with separate edges + weights arrays.

    Args:
        n (int)
        edges: list of (i, j)
        weights: list of floats, same length as `edges`
    """

    model = Model("maxcut")
    model.setParam("OutputFlag", 1 if verbose else 0)
    if time_limit is not None:
        model.setParam("TimeLimit", time_limit)
    if mip_gap is not None:
        model.setParam("MIPGap", mip_gap)

    # Vars
    y = model.addVars(n, vtype=GRB.BINARY, name="y")
    z = model.addVars(len(edges), vtype=GRB.BINARY, name="z")

    # Constraints: z[k] = |y_i - y_j|
    for k, (i, j) in enumerate(edges):
        model.addConstr(z[k] >= y[i] - y[j])
        model.addConstr(z[k] >= y[j] - y[i])
        model.addConstr(z[k] <= y[i] + y[j])
        model.addConstr(z[k] <= 2 - (y[i] + y[j]))

    # Objective
    model.setObjective(quicksum(weights[k] * z[k] for k in range(len(edges))),
                       GRB.MAXIMIZE)

    model.optimize()

    if model.SolCount == 0:
        return None, None, None

    obj = model.ObjVal
    y_sol = [int(y[i].X) for i in range(n)]
    z_sol = [int(z[k].X) for k in range(len(edges))]

    return obj, y_sol, z_sol

In [11]:
rounded_solution = round_sdp_with_cholesky(M_optimal)

print(rounded_solution)
edge_count = 0
for i in rounded_solution:
    for j in rounded_solution:
        if i!=j and (i, j) in edges:
            edge_count += 1

print(edge_count)


[0 0 0 0 1 1 1 0 0 0 1 0 0 1 0 0 0 0 0 0 1 1 1 1 1 0 0 1 1 0 1 0 1 0 0 0 0
 0 1 0 0 0 1 1 1 0 0 1 0 0]
1178


In [ ]:
print(gurobi_maxcut(n_vertices, edges, weights))

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2732718
Academic license 2732718 - for non-commercial use only - registered to j.___@student.maastrichtuniversity.nl
Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Academic license 2732718 - for non-commercial use only - registered to j.___@student.maastrichtuniversity.nl
Optimize a model with 8824 rows, 2256 columns and 26472 nonzeros (Max)
Model fingerprint: 0xf622ff27
Model has 2206 linear objective coefficients
Variable types: 0 continuous, 2256 integer (2256 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 2e+00]
Found heuristic solution: objective -0.0000000
Presolve removed 6412 rows and 1000 columns
Presolve time: 0